# Walkthrough: from ECB curve parameters to what the market prices

Run `erm fetch` first. Every cell reads the local cache in `data/raw/`; nothing here
touches the network. Each step is a check you could be asked to do on a whiteboard.

In [ ]:
import numpy as np
import pandas as pd

from euro_rates_monitor import build, curve, forwards, load, pca

spot = load.ecb_curve("ecb_spot_aaa")
params = load.ecb_wide("ecb_params_aaa").dropna()
day = spot.index[-1]
spot.tail(3).round(3)

## 1. The published curve is a Svensson fit, and we can reproduce it exactly

The ECB publishes six parameters per day. Evaluating the Svensson formula at the
published maturities must return the published spot rates.

In [ ]:
ours = curve.spot_from_params(params.loc[day], np.array(spot.columns))
pd.DataFrame({"published": spot.loc[day], "from parameters": ours,
              "diff (bp)": (ours - spot.loc[day]) * 100}).round(6)

## 2. One forward rate by hand

Continuous compounding: f(t1, t2) = (r2·t2 − r1·t1) / (t2 − t1). The 1Y rate in 1Y
from the 1Y and 2Y spot rates:

In [ ]:
r1, r2 = spot.loc[day, 1.0], spot.loc[day, 2.0]
by_hand = (r2 * 2 - r1 * 1) / (2 - 1)
print(f"1Y spot {r1:.4f}%, 2Y spot {r2:.4f}%  ->  1Y1Y forward {by_hand:.4f}%")
print(f"library: {float(forwards.forward_rate(r1, 1.0, r2, 2.0)):.4f}%")

## 3. Short forwards converge to the instantaneous forward

The ECB technical note gives the instantaneous forward in closed form:
f(t) = b0 + b1·e^(−t/τ1) + b2·(t/τ1)·e^(−t/τ1) + b3·(t/τ2)·e^(−t/τ2).
As the forward period shrinks, our discrete forwards must approach it.

In [ ]:
p = params.loc[day]
def inst_fwd(t):
    return (p.BETA0 + p.BETA1 * np.exp(-t / p.TAU1) + p.BETA2 * (t / p.TAU1) * np.exp(-t / p.TAU1)
            + p.BETA3 * (t / p.TAU2) * np.exp(-t / p.TAU2))
t = 1.0
for dt in (0.25, 0.05, 0.001):
    r_a, r_b = curve.spot_from_params(p, np.array([t, t + dt]))
    print(f"period {dt:>5}: discrete {float(forwards.forward_rate(r_a, t, r_b, t + dt)):.5f}%"
          f"   instantaneous {inst_fwd(t):.5f}%")

## 4. The path and its sensitivity to a term premium

Headline: raw 3M forwards. Scenario columns subtract k bp per year of horizon, an
illustration of how much a term premium could move the reading, not an estimate.

In [ ]:
path = forwards.term_premium_scenarios(forwards.forward_path(p))
path.round(3)

## 5. PCA: level, slope, curvature

In [ ]:
fit = pca.fit_pca(spot)
display(fit.explained.mul(100).round(1).rename("% of variance"))
display(fit.loadings.round(2))
{name: pca.sign_changes(fit.loadings[name].to_numpy()) for name in fit.loadings}

Curve-shape anomaly (residual after three factors). On a fitted Svensson curve this is
**not** a tradable rich/cheap signal; it only shows where the fitted shape is unusual.

In [ ]:
pca.anomaly_table(spot, fit).round(2)

## 6. Everything the note is allowed to say

In [ ]:
res = build.run()
pd.Series({k: v for k, v in res.metrics.items() if not isinstance(v, list)}).to_frame("value")